# 1.data_process_15test.py

Autogenerated from the matching source file in `14.15Test/code_src`.

In [1]:
from pathlib import Path
import warnings

import networkx as nx
import numpy as np
import pandas as pd
from fastdtw import fastdtw
from numpy.linalg import norm
from ripser import ripser
from scipy.stats import wasserstein_distance
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

ROOT = Path("/Users/jane/Documents/202511吾-Systems/14.15Test")
SIM_DIR = ROOT / "similarity" / "21_1"
TDA_DIR = ROOT / "tda_results" / "21_1"
MERGED_DIR = ROOT / "merged"
TABLE_DIR = ROOT / "tables"
PLOT_DIR = ROOT / "plots"
INPUT_DIR = ROOT / "inputs"

for path in [SIM_DIR, TDA_DIR, MERGED_DIR, TABLE_DIR, PLOT_DIR]:
    path.mkdir(parents=True, exist_ok=True)

WINDOW_SIZE = 21
LAG = 1
N_EPS = 50

EVENTS = pd.to_datetime([
    "2020-12-01", "2021-01-02", "2021-01-07", "2021-01-29", "2021-02-16",
    "2021-03-13", "2021-04-10", "2021-05-12", "2021-05-17", "2021-05-18",
    "2021-05-19", "2021-06-09", "2021-09-24", "2021-10-15", "2021-11-15",
    "2022-04-27", "2022-05-01", "2022-05-11", "2022-05-12", "2022-05-13",
    "2022-07-20", "2022-11-01", "2023-03-01", "2023-03-10", "2023-05-17",
    "2023-06-16", "2023-07-01", "2023-10-01", "2024-03-19", "2024-04-20",
    "2025-01-20", "2025-02-03", "2025-02-21", "2025-03-07", "2025-05-20",
    "2025-06-05", "2025-06-17",
])


def time_delay_embedding(series, window_size, lag):
    series = np.asarray(series, dtype=float)
    n = len(series)
    m = n - (window_size - 1) * lag
    if m <= 0:
        raise ValueError("Series too short for embedding.")
    out = np.empty((m, window_size), dtype=float)
    for i in range(m):
        out[i, :] = series[i:(i + window_size * lag):lag]
    return out


def build_embedding_panel(df, value_cols, date_col="Date", window_size=21, lag=1):
    dates = pd.to_datetime(df[date_col]).reset_index(drop=True)
    values = df[value_cols].astype(float).reset_index(drop=True)
    embedded = {
        col: time_delay_embedding(values[col].to_numpy(), window_size, lag)
        for col in value_cols
    }
    end_dates = dates.iloc[(window_size - 1) * lag:].reset_index(drop=True)
    return embedded, end_dates


def vector_dist(u, v):
    u = np.ravel(np.array(u, dtype=float))
    v = np.ravel(np.array(v, dtype=float))
    return norm(u - v)


def compute_similarity_stack(embedded_dict, coins, method):
    n_coins = len(coins)
    n_time = embedded_dict[coins[0]].shape[0]
    stack = np.zeros((n_time, n_coins, n_coins), dtype=float)
    for t in tqdm(range(n_time), desc=f"Computing {method}"):
        dist_mat = np.zeros((n_coins, n_coins), dtype=float)
        for i in range(n_coins):
            for j in range(i + 1, n_coins):
                seq_i = embedded_dict[coins[i]][t, :]
                seq_j = embedded_dict[coins[j]][t, :]
                if method == "wasserstein":
                    d = wasserstein_distance(seq_i, seq_j)
                elif method == "dtw":
                    d, _ = fastdtw(seq_i, seq_j, dist=vector_dist)
                else:
                    raise ValueError(method)
                dist_mat[i, j] = d
                dist_mat[j, i] = d
        stack[t] = dist_mat
    return stack


def betti_numbers(diagrams, thresholds):
    betti_0, betti_1 = [], []
    d0 = diagrams[0] if len(diagrams) > 0 else np.empty((0, 2))
    d1 = diagrams[1] if len(diagrams) > 1 else np.empty((0, 2))
    for eps in thresholds:
        b0 = np.sum((d0[:, 0] <= eps) & (d0[:, 1] > eps)) if len(d0) else 0
        b1 = np.sum((d1[:, 0] <= eps) & (d1[:, 1] > eps)) if len(d1) else 0
        betti_0.append(b0)
        betti_1.append(b1)
    return np.asarray(betti_0, dtype=float), np.asarray(betti_1, dtype=float)


def persistent_entropy(diagram):
    if diagram.size == 0:
        return 0.0
    lifetimes = diagram[:, 1] - diagram[:, 0]
    lifetimes = lifetimes[np.isfinite(lifetimes) & (lifetimes > 0)]
    if len(lifetimes) == 0:
        return 0.0
    p = lifetimes / lifetimes.sum()
    return float(-(p * np.log(p)).sum())


def compute_tda_summary(similarity_stack, n_eps=50):
    stack = np.asarray(similarity_stack, dtype=float)
    eps_max = float(np.nanmax(stack))
    epsilons = np.linspace(0, eps_max, n_eps)
    n_time = stack.shape[0]
    betti0_series = np.zeros((n_time, n_eps), dtype=float)
    betti1_series = np.zeros((n_time, n_eps), dtype=float)
    entropy_series = np.zeros(n_time, dtype=float)
    for t in tqdm(range(n_time), desc="Computing TDA"):
        dist_mat = np.nan_to_num(stack[t].copy(), nan=0.0, posinf=0.0, neginf=0.0)
        dist_mat = (dist_mat + dist_mat.T) / 2.0
        np.fill_diagonal(dist_mat, 0.0)
        dgms = ripser(dist_mat, distance_matrix=True, maxdim=1)["dgms"]
        b0, b1 = betti_numbers(dgms, epsilons)
        betti0_series[t] = b0
        betti1_series[t] = b1
        entropy_series[t] = persistent_entropy(dgms[1])
    return {
        "epsilons": epsilons,
        "betti0_series": betti0_series,
        "betti1_series": betti1_series,
        "entropy_series": entropy_series,
    }


def distance_to_weight(dist_mat):
    d = np.nan_to_num(np.asarray(dist_mat, dtype=float), nan=0.0, posinf=0.0, neginf=0.0)
    d = (d + d.T) / 2.0
    np.fill_diagonal(d, 0.0)
    return 1.0 / (1.0 + d)


def compute_network_metrics(stack, prefix, dates):
    rows = []
    for idx, dist_mat in enumerate(stack):
        w = distance_to_weight(dist_mat)
        g = nx.from_numpy_array(w)
        clustering = float(np.mean(list(nx.clustering(g, weight="weight").values())))
        degree_strength = w.sum(axis=1) / (w.shape[0] - 1)
        degree_centrality = float(degree_strength.mean())
        pagerank_max = float(max(nx.pagerank(g, weight="weight").values()))
        rows.append({
            "time_index": idx,
            "date": dates.iloc[idx],
            f"{prefix}_clustering": clustering,
            f"{prefix}_degree_centrality": degree_centrality,
            f"{prefix}_pagerank": pagerank_max,
        })
    return pd.DataFrame(rows)


print("Loading inputs...")
close_df = pd.read_csv(ROOT / "Close_price_15.csv")
park_df = pd.read_csv(ROOT / "Parkinson_volatility_15.csv")
rogers_df = pd.read_csv(INPUT_DIR / "Rogers_volatility_15.csv")
market_df = pd.read_csv(INPUT_DIR / "SP500_VIX_BVOL_CVI.csv")
fng_df = pd.read_csv(INPUT_DIR / "fear_and_greed_index.csv")

for df in [close_df, park_df, rogers_df]:
    df["Date"] = pd.to_datetime(df["Date"])
    df.sort_values("Date", inplace=True)
    df.reset_index(drop=True, inplace=True)

market_df["Date"] = pd.to_datetime(market_df["Date"])
fng_df["Date"] = pd.to_datetime(fng_df["Date"])
market_df = market_df.sort_values("Date").reset_index(drop=True)
fng_df = fng_df.sort_values("Date").reset_index(drop=True)

coins = [c for c in close_df.columns if c != "Date"]

log_df = close_df.copy()
log_df[coins] = np.log(log_df[coins].astype(float) / log_df[coins].astype(float).shift(1))
log_df = log_df.dropna().reset_index(drop=True)

log_emb, log_dates = build_embedding_panel(log_df, coins, window_size=WINDOW_SIZE, lag=LAG)
vol_emb, vol_dates = build_embedding_panel(park_df, coins, window_size=WINDOW_SIZE, lag=LAG)

common_dates = pd.Index(log_dates)
vol_offset = len(vol_dates) - len(common_dates)
if vol_offset < 0:
    raise ValueError("Unexpected date alignment issue: vol_dates shorter than log_dates.")
vol_dates = vol_dates.iloc[vol_offset:].reset_index(drop=True)
vol_emb = {k: v[vol_offset:] for k, v in vol_emb.items()}
assert pd.Index(vol_dates).equals(common_dates)

embedding_manifest = pd.DataFrame([
    {
        "panel": "log_return",
        "n_time": len(log_dates),
        "n_coins": len(coins),
        "window_size": WINDOW_SIZE,
        "lag": LAG,
        "start_date": log_dates.iloc[0],
        "end_date": log_dates.iloc[-1],
    },
    {
        "panel": "parkinson_volatility",
        "n_time": len(vol_dates),
        "n_coins": len(coins),
        "window_size": WINDOW_SIZE,
        "lag": LAG,
        "start_date": vol_dates.iloc[0],
        "end_date": vol_dates.iloc[-1],
    },
])
embedding_manifest.to_csv(MERGED_DIR / "embedding_manifest_15.csv", index=False)

pd.DataFrame({"event_date": EVENTS}).to_csv(MERGED_DIR / "crypto_market_events.csv", index=False)

method_specs = [
    ("vol_wass", "volatility", "wasserstein", vol_emb, vol_dates, SIM_DIR / "vol_wasserstein_similarity.npz"),
    ("log_wass", "log_return", "wasserstein", log_emb, log_dates, SIM_DIR / "log_wasserstein_similarity.npz"),
    ("log_dtw", "log_return", "dtw", log_emb, log_dates, SIM_DIR / "log_DTW_similarity.npz"),
    ("vol_dtw", "volatility", "dtw", vol_emb, vol_dates, SIM_DIR / "vol_DTW_similarity.npz"),
]

similarity_results = {}
for prefix, value_type, method, emb_dict, dates, out_path in method_specs:
    if out_path.exists():
        data = np.load(out_path, allow_pickle=True)
        stack = data["similarity_matrices"]
        print(f"Loaded cached similarity: {out_path.name}")
    else:
        stack = compute_similarity_stack(emb_dict, coins, method=method)
        np.savez(
            out_path,
            similarity_matrices=stack,
            dates=dates.astype(str).to_numpy(),
            coins=np.array(coins),
            value_type=value_type,
            method=method,
            window_size=WINDOW_SIZE,
            lag=LAG,
        )
        print(f"Saved similarity: {out_path.name}")
    similarity_results[prefix] = stack

similarity_manifest = pd.DataFrame([
    {
        "prefix": prefix,
        "value_type": value_type,
        "method": method,
        "path": str(out_path),
        "n_time": similarity_results[prefix].shape[0],
        "n_coins": similarity_results[prefix].shape[1],
    }
    for prefix, value_type, method, emb_dict, dates, out_path in method_specs
])
similarity_manifest.to_csv(MERGED_DIR / "similarity_manifest_15.csv", index=False)

prefix_to_triplet = {
    "vol_wass": ("vol", "wass"),
    "log_wass": ("log", "wass"),
    "log_dtw": ("log", "dtw"),
    "vol_dtw": ("vol", "dtw"),
}

tda_csv_frames = []
for prefix, stack in similarity_results.items():
    out_npz = TDA_DIR / f"{prefix}_tda_summary.npz"
    if out_npz.exists():
        data = np.load(out_npz, allow_pickle=True)
        summary = {
            "epsilons": data["epsilons"],
            "betti0_series": data["betti0_series"],
            "betti1_series": data["betti1_series"],
            "entropy_series": data["entropy_series"],
        }
        print(f"Loaded cached TDA: {out_npz.name}")
    else:
        summary = compute_tda_summary(stack, n_eps=N_EPS)
        np.savez(out_npz, **summary)
        print(f"Saved TDA: {out_npz.name}")

    left, right = prefix_to_triplet[prefix]
    df_out = pd.DataFrame({
        "time_index": np.arange(len(log_dates)),
        "date": log_dates,
        f"{left}_{right}_betti0": summary["betti0_series"].mean(axis=1),
        f"{left}_{right}_betti1": summary["betti1_series"].mean(axis=1),
        f"{left}_{right}_entropy": summary["entropy_series"],
    })
    csv_path = TDA_DIR / f"{prefix}_tda_features.csv"
    df_out.to_csv(csv_path, index=False)
    tda_csv_frames.append(df_out)

merged_tda = pd.DataFrame({"time_index": np.arange(len(log_dates)), "date": log_dates})
for df_part in tda_csv_frames:
    merged_tda = merged_tda.merge(df_part, on=["time_index", "date"], how="left")
merged_tda.to_csv(MERGED_DIR / "TDA_results_merged_21_1.csv", index=False)

network_parts = []
for prefix, stack in similarity_results.items():
    csv_path = MERGED_DIR / f"{prefix}_network_metrics.csv"
    if csv_path.exists():
        net_df = pd.read_csv(csv_path, parse_dates=["date"])
    else:
        net_df = compute_network_metrics(stack, prefix, log_dates)
        net_df.to_csv(csv_path, index=False)
    network_parts.append(net_df)

network_df = pd.DataFrame({"time_index": np.arange(len(log_dates)), "date": log_dates})
for part in network_parts:
    network_df = network_df.merge(part, on=["time_index", "date"], how="left")
network_df.to_csv(MERGED_DIR / "network_metrics_from_similarity_15.csv", index=False)

btc_park = park_df[["Date", "BTC"]].rename(columns={"Date": "date", "BTC": "vol_btc_parkinson"})
btc_rogers = rogers_df[["Date", "BTC"]].rename(columns={"Date": "date", "BTC": "vol_btc_rogers"})
market_small = market_df.rename(columns={"Date": "date"})
fng_small = fng_df.rename(columns={"Date": "date"})

merged_panel = (
    merged_tda
    .merge(network_df, on=["time_index", "date"], how="left")
    .merge(btc_park, on="date", how="left")
    .merge(btc_rogers, on="date", how="left")
    .merge(market_small, on="date", how="left")
    .merge(fng_small, on="date", how="left")
)
merged_panel["fear&greed"] = merged_panel["fng_value"]
merged_panel["fear_greed"] = merged_panel["fng_value"]
merged_panel.to_csv(MERGED_DIR / "TDA_SP500_VIX_BTC_BVOL_CVI_15_merged_with_network.csv", index=False)

labeled_panel = merged_panel.copy()
labeled_panel["event_label"] = 0
for e in EVENTS:
    labeled_panel.loc[
        (labeled_panel["date"] >= e - pd.Timedelta(days=21))
        & (labeled_panel["date"] <= e + pd.Timedelta(days=21)),
        "event_label",
    ] = 1
labeled_panel.to_csv(MERGED_DIR / "TDA_SP500_VIX_BTC_BVOL_CVI_15_merged_labeled_with_network.csv", index=False)

manifest = pd.DataFrame([
    {"file": "TDA_results_merged_21_1.csv", "path": str(MERGED_DIR / "TDA_results_merged_21_1.csv")},
    {"file": "network_metrics_from_similarity_15.csv", "path": str(MERGED_DIR / "network_metrics_from_similarity_15.csv")},
    {"file": "TDA_SP500_VIX_BTC_BVOL_CVI_15_merged_with_network.csv", "path": str(MERGED_DIR / "TDA_SP500_VIX_BTC_BVOL_CVI_15_merged_with_network.csv")},
    {"file": "TDA_SP500_VIX_BTC_BVOL_CVI_15_merged_labeled_with_network.csv", "path": str(MERGED_DIR / "TDA_SP500_VIX_BTC_BVOL_CVI_15_merged_labeled_with_network.csv")},
    {"file": "crypto_market_events.csv", "path": str(MERGED_DIR / "crypto_market_events.csv")},
])
manifest.to_csv(MERGED_DIR / "pipeline_manifest_15.csv", index=False)

print("Data process finished.")
print("TDA shape:", merged_tda.shape)
print("Network shape:", network_df.shape)
print("Merged labeled shape:", labeled_panel.shape)


Loading inputs...


Computing wasserstein:   0%|          | 0/1853 [00:00<?, ?it/s]

Saved similarity: vol_wasserstein_similarity.npz


Computing wasserstein:   0%|          | 0/1853 [00:00<?, ?it/s]

Saved similarity: log_wasserstein_similarity.npz


...

Computing TDA:   0%|          | 0/1853 [00:00<?, ?it/s]

Saved TDA: log_dtw_tda_summary.npz


Computing TDA:   0%|          | 0/1853 [00:00<?, ?it/s]

Saved TDA: vol_dtw_tda_summary.npz


Data process finished.
TDA shape: (1853, 14)
Network shape: (1853, 14)
Merged labeled shape: (1853, 37)
